Project: Customer-Support Chatbot for an E-Commerce store

In [26]:
# Import standard libraries for file handling and text processing
import os, pathlib, textwrap, glob

# Load documents from various sources (URLs, text files, PDFs)
from langchain_community.document_loaders import UnstructuredURLLoader, TextLoader, PyPDFLoader

# Split long texts into smaller, manageable chunks for embedding
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Vector store to store and retrieve embeddings efficiently using FAISS
from langchain.vectorstores import FAISS

# Generate text embeddings using OpenAI or Hugging Face models
from langchain.embeddings import OpenAIEmbeddings, HuggingFaceEmbeddings, SentenceTransformerEmbeddings

# Use local LLMs (e.g., via Ollama) for response generation
from langchain.llms import Ollama

# Build a retrieval chain that combines a retriever, a prompt, and an LLM


# Create prompts for the RAG system


print("Libraries imported - you're good to go!")

Libraries imported - you're good to go!


2 Data preparation

2.1 Ingest source documents

In [ ]:
pdf_paths= glob.glob("data/Everstorm_*.pdf")
raw_docs=[]

# tmp=PyPDFLoader(pdf_paths[0]).load()
# print(tmp)
# print(tmp[0].page_content)
for path in pdf_paths:
    tmp= PyPDFLoader(path).load()
    raw_docs.extend(tmp)

print(f"Loaded {len(raw_docs)} PDF pages from {len(pdf_paths)} files.")
raw_docs[0]

Ignoring wrong pointing object 76 0 (offset 0)
Ignoring wrong pointing object 81 0 (offset 0)
Ignoring wrong pointing object 80 0 (offset 0)


Loaded 8 PDF pages from 4 files.


Document(metadata={'producer': 'Skia/PDF m138 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Payment_refund_and_security', 'source': 'data\\Everstorm_Payment_refund_and_security.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='FAQs  \n Which  payment  methods  do  you  accept?   Visa,  MasterCard,  AmEx,  Discover,  Apple  Pay,  Google  Pay,  PayPal,  Shop  Pay.  Installments  (US  \nonly),\n \nKlarna\n \nPay-in-4\n \n(selected\n \nEU\n \ncountries).\n  What  is  3-D  Secure  and  why  did  I  see  a  pop-up?    3-D  Secure  (also  “Verified  by  Visa”  /  “Mastercard  Identity  Check”)  adds  an  extra  one-time  code  \nfor\n \nEU\n \nPSD2\n \ncompliance.\n \nYour\n \nbank\n \ncontrols\n \nthat\n \npop-up.\n  Is  my  data  safe?   All  checkout  traffic  uses  TLS  1.3.  We  never  store  full  card  numbers.  Our  store  is  Level  1  \nPCI-DSS\n \ncompliant.\n  How  long  do  refunds  take?    We  issue  refunds  the  same  day  your  r

2.1optional- load from web pages

In [ ]:
URLS = [
    # --- BigCommerce – shipping & refunds ---
    "https://developer.bigcommerce.com/docs/store-operations/shipping",
    "https://developer.bigcommerce.com/docs/store-operations/orders/refunds",
    # --- Stripe – disputes & chargebacks ---
    # "https://docs.stripe.com/disputes",
    # --- WooCommerce – REST API reference ---
    # "https://woocommerce.github.io/woocommerce-rest-api-docs/v3.html",
]

try:
    from langchain_community.document_loaders import WebBaseLoader

    loader = WebBaseLoader(URLS)
    raw_docs = loader.load()
    print(f"Fetched {len(raw_docs)} documents from the web.")
except Exception as e:
    print("⚠️  Web fetch failed, using offline copies:", e)
    raw_docs = []

    from langchain_community.document_loaders import DirectoryLoader, TextLoader

    # Put any fallback files under ./offline_docs (e.g., .md, .txt, .html)
    patterns = ["**/*.md", "**/*.txt", "**/*.html"]
    for pattern in patterns:
        try:
            loader = DirectoryLoader(
                "offline_docs",
                glob=pattern,
                loader_cls=TextLoader,
                show_progress=True,
                use_multithreading=True,
            )
            raw_docs.extend(loader.load())
        except Exception:
            # Skip unreadable pattern/file types quietly
            pass

    print(f"Loaded {len(raw_docs)} offline documents.")

Fetched 2 documents from the web.


2.2 Chunk the text

In [ ]:
chunks=[]
text_splitter= RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks= text_splitter.split_documents(raw_docs)
print(f"{len(chunks)} chunks ready for embedding.")

42 chunks ready for embedding.


3 Build a retriever 

3.1 Load a model to generate embeddings

In [ ]:
embedding_vector=[]
embeddings= SentenceTransformerEmbeddings(model_name="thenlper/gte-small")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [ ]:
text= "Hello world"
embedding_vector= embeddings.embed_query(text)
print(embedding_vector)
print(len(embedding_vector))

[-0.03076753206551075, 0.0007690953207202256, 0.03574733808636665, -0.05139586701989174, 0.011312486603856087, -0.002577443141490221, 0.003908256068825722, 0.04378264397382736, 0.007158250082284212, -0.022576769813895226, -0.0015379164833575487, -0.06957899779081345, 0.040758293122053146, 0.054925691336393356, -0.021120863035321236, -0.017361687496304512, -0.018123282119631767, -0.009675391018390656, -0.09127013385295868, 0.011706655845046043, 0.08448991924524307, -0.0115936528891325, -0.009450915269553661, -0.04615940526127815, -0.008405952714383602, 0.02692677453160286, -0.015226863324642181, -0.011262929998338223, -0.013487381860613823, -0.18782880902290344, -0.021480534225702286, -0.03052482381463051, 0.0635928362607956, -0.024447552859783173, 0.027081573382019997, -0.029036249965429306, -0.0250388290733099, 0.0421237088739872, -0.024005509912967682, 0.03671465069055557, 0.011428236961364746, -0.0365777388215065, -0.015722721815109253, -0.06126299127936363, -9.054446127265692e-05, 

3.2 Build a vector database

In [ ]:
vectordb= FAISS.from_documents(chunks, embeddings)
retriever = vectordb.as_retriever(search_kwargs={"k":8})
print("Vector store with", vectordb.index.ntotal, "embeddings")

Vector store with 42 embeddings


4 Build the generation engine

4.1 Install ollama and serve gemma3

4.2 test an LLM with a random prompt

In [ ]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="gemma3:1b"
)

response = llm.invoke("Explain RAG in one sentence")

print(response)

c:\Users\freny\miniconda3\envs\rag-chatbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RAG (Retrieval- Augmented Generation)—often referred to as "shotgun chatting" – involves retrieving relevant information from an external source and feeding it along with a question, allowing the model to generate a more complete and informed response.



Build a RAG

5.1 Define a system prompt

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.chains import ConversationalRetrievalChain

SYSTEM_TEMPLATE = """
You are a **Customer Support Chatbot**. Use only the information in CONTEXT to answer.
If the answer is not in CONTEXT, respond with “I'm not sure from the docs.”

Rules:
1) Use ONLY the provided <context> to answer.
2) If the answer is not in the context, say: "I don't know based on the retrieved documents."
3) Be concise and accurate. Prefer quoting key phrases from the context.
4) When possible, cite sources as [source: source] using the metadata.

CONTEXT:
{context}

USER:
{question}
"""

5.2 Create a RAG chain

In [ ]:
prompt= PromptTemplate(input_variables=["context", "question"], template=SYSTEM_TEMPLATE)
llm = OllamaLLM(
    model="gemma3:1b", temperature=0.1
)
chain = ConversationalRetrievalChain.from_llm(llm, retriever, combine_docs_chain_kwargs={"prompt": prompt}, return_source_documents=True)

5.3 Validate the RAG chain

In [ ]:
test_questions = [
    "If I'm not happy with my purchase, what is your refund policy and how do I start a return?",
    "How long will delivery take for a standard order, and where can I track my package once it ships?",
    "What's the quickest way to contact your support team, and what are your operating hours?",
]

chat_history = []
for q in test_questions:
    result = chain({"question": q, "chat_history": chat_history})
    chat_history.append((q, result["answer"]))
    print(f"\nQ: {q}\nA: {result['answer'][:350]}...")